In [ ]:
import tensorflow as tf
import keras
import sys


print("Python :", sys.executable)
print("TensorFlow :", tf.__version__)
print("Keras :", keras.__version__)

In [2]:
# Pre

In [ ]:
import os
import shutil
import pandas as pd

from sklearn.model_selection import train_test_split

base_dir = r'C:\Users\ASUS\Desktop\j\ears\data'

csv_path = r'C:\Users\ASUS\Desktop\j\ears\IR-EAR dataset .csv'

ears_folder = r'C:\Users\ASUS\Desktop\j\ears\updatedMainDataset_ear'

TRAIN_SIZE = 0.80
VAL_SIZE = 0.10
TEST_SIZE = 0.10

RANDOM_STATE = 42

df = pd.read_csv(csv_path)

print(df.head())

print("\nColumns:")
print(df.columns.tolist())

gender_check = (
    df.groupby('ID')['Gender']
      .nunique()
)

invalid_ids = gender_check[
    gender_check > 1
].index.tolist()

if len(invalid_ids) > 0:

    print("\nError!")

    print(
        df[df['ID'].isin(invalid_ids)]
        [['ID', 'Gender']]
        .sort_values('ID')
    )

    raise ValueError(
        f"found {len(invalid_ids)} person-ID."
    )

else:
    print("All True")

required_columns = [
    'ID',
    'ID-Right',
    'ID-Left',
    'Gender'
]

for col in required_columns:

    if col not in df.columns:

        raise ValueError(
            f"'{col}' not found in file"
        )

df = df[
    ['ID', 'ID-Right', 'ID-Left', 'Gender']
].copy()

df = df.dropna(
    subset=['ID', 'Gender']
)

# Man->0
# Woman->1

df['Gender'] = (
    df['Gender']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'man': 0,
        'woman': 1
    })
)

if df['Gender'].isna().any():

    unknown_values = df.loc[
        df['Gender'].isna(),
        'Gender'
    ]

    raise ValueError(
        "unknown values"
    )


df['Gender'] = df['Gender'].astype(int)

if not df['Gender'].isin([0, 1]).all():

    raise ValueError(
        "gender column must be only 0 and 1."
    )

print("\ngender distribution:")
print(df['Gender'].value_counts().sort_index())

for split in [
    'train',
    'validation',
    'test'
]:

    for gender in [
        '0',
        '1'
    ]:

        path = os.path.join(
            base_dir,
            split,
            gender
        )

        os.makedirs(
            path,
            exist_ok=True
        )

# person level table

persons = (
    df[
        ['ID', 'Gender']
    ]
    .drop_duplicates(
        subset='ID'
    )
    .copy()
)


print(
    "Total persons:",
    len(persons)
)

print(
    "Male persons:",
    (persons['Gender'] == 0).sum()
)

print(
    "Female persons:",
    (persons['Gender'] == 1).sum()
)

person_ids = persons['ID'].values

person_gender = persons['Gender'].values


# train=>80%
# test+validation=>20%

train_ids, temp_ids = train_test_split(

    person_ids,

    test_size=VAL_SIZE + TEST_SIZE,

    random_state=RANDOM_STATE,

    stratify=person_gender
)

temp_persons = persons[
    persons['ID'].isin(temp_ids)
].copy()


val_ids, test_ids = train_test_split(

    temp_persons['ID'].values,

    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),

    random_state=RANDOM_STATE,

    stratify=temp_persons['Gender']
)

print(
    "Train persons:",
    len(train_ids)
)

print(
    "Validation persons:",
    len(val_ids)
)

print(
    "Test persons:",
    len(test_ids)
)

train_persons = persons[
    persons['ID'].isin(train_ids)
]

val_persons = persons[
    persons['ID'].isin(val_ids)
]

test_persons = persons[
    persons['ID'].isin(test_ids)
]

print("\nTRAIN:")
print(
    train_persons['Gender']
    .value_counts()
    .sort_index()
)

print("\nVALIDATION:")
print(
    val_persons['Gender']
    .value_counts()
    .sort_index()
)

print("\nTEST:")
print(
    test_persons['Gender']
    .value_counts()
    .sort_index()
)

def copy_ear_image(
    image_id,
    gender,
    split
):

    if pd.isna(image_id):

        return


    image_id = int(image_id)


    filename = (
        f'{image_id}.png'
    )


    source = os.path.join(
        ears_folder,
        filename
    )


    destination = os.path.join(

        base_dir,

        split,

        str(gender),

        filename
    )


    if os.path.exists(source):

        shutil.copy2(
            source,
            destination
        )

    else:

        print(
            "file not found: ",
            source
        )

for _, row in df.iterrows():

    person_id = row['ID']

    gender = int(row['Gender'])

    if person_id in train_ids:

        split = 'train'

    elif person_id in val_ids:

        split = 'validation'

    elif person_id in test_ids:

        split = 'test'

    else:

        continue

    copy_ear_image(

        row['ID-Right'],

        gender,

        split
    )

    copy_ear_image(

        row['ID-Left'],

        gender,

        split
    )


for split in [
    'train',
    'validation',
    'test'
]:

    for gender in [
        '0',
        '1'
    ]:

        folder = os.path.join(

            base_dir,

            split,

            gender
        )


        count = len([

            f

            for f in os.listdir(folder)

            if f.lower().endswith('.png')

        ])


        print(
            f'{split:12s} Gender {gender}: {count}'
        )

In [ ]:
train_persons = persons[
    persons['ID'].isin(train_ids)
].copy()

val_persons = persons[
    persons['ID'].isin(val_ids)
].copy()

test_persons = persons[
    persons['ID'].isin(test_ids)
].copy()

print(
    f"Train      : {len(train_persons)} persons"
)

print(
    f"Validation : {len(val_persons)} persons"
)

print(
    f"Test       : {len(test_persons)} persons"
)
def print_person_gender_counts(name, dataframe):

    male = (dataframe['Gender'] == 0).sum()
    female = (dataframe['Gender'] == 1).sum()

    print(f"\n{name}")
    print(f"Male: {male}")
    print(f"Female: {female}")
    print(f"Total: {len(dataframe)}")


print_person_gender_counts(
    "TRAIN",
    train_persons
)

print_person_gender_counts(
    "VALIDATION",
    val_persons
)

print_person_gender_counts(
    "TEST",
    test_persons
)

In [ ]:
print(df[['ID', 'ID-Right', 'ID-Left', 'Gender']].head(20))
print(df['Gender'].value_counts())

In [ ]:
import os
import pandas as pd
import tensorflow as tf

base_dir = r'C:\Users\ASUS\Desktop\j\ears\data'


def build_dataframe(split_name):

    image_paths = []
    labels = []

    split_path = os.path.join(
        base_dir,
        split_name
    )

    for gender in ["0", "1"]:

        class_path = os.path.join(
            split_path,
            gender
        )

        if not os.path.exists(class_path):
            continue

        for file in sorted(os.listdir(class_path)):

            if file.lower().endswith(
                (".png", ".jpg", ".jpeg")
            ):

                image_paths.append(
                    os.path.join(
                        class_path,
                        file
                    )
                )

                labels.append(
                    int(gender)
                )

    return pd.DataFrame({
        "path": image_paths,
        "gender": labels
    })


train_df = build_dataframe("train")
val_df = build_dataframe("validation")
test_df = build_dataframe("test")

train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

val_df = val_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

test_df = test_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("=" * 60)
print("dataframe info")
print("=" * 60)

print("Train Images:", len(train_df))
print("validation Images :", len(val_df))
print("test Images:", len(test_df))

print()

print("Train Distribution:")
print(train_df["gender"].value_counts().sort_index())

print()

print("Validation Distribution:")
print(val_df["gender"].value_counts().sort_index())

print()

print("Test Distribution:")
print(test_df["gender"].value_counts().sort_index())

In [9]:
from tensorflow.keras.applications.vgg16 import (
    preprocess_input as vgg16_preprocess
)

from tensorflow.keras.applications.vgg19 import (
    preprocess_input as vgg19_preprocess
)

from tensorflow.keras.applications.resnet50 import (
    preprocess_input as resnet50_preprocess
)

from tensorflow.keras.applications.densenet import (
    preprocess_input as densenet_preprocess
)

from tensorflow.keras.applications.xception import (
    preprocess_input as xception_preprocess
)

from tensorflow.keras.applications.inception_v3 import (
    preprocess_input as inception_v3_preprocess
)

from tensorflow.keras.applications.inception_resnet_v2 import (
    preprocess_input as inception_resnet_v2_preprocess
)

PREPROCESS_FUNCTIONS = {

    "VGG16":
        vgg16_preprocess,

    "VGG19":
        vgg19_preprocess,

    "ResNet50":
        resnet50_preprocess,

    "DenseNet121":
        densenet_preprocess,

    "DenseNet169":
        densenet_preprocess,

    "DenseNet201":
        densenet_preprocess,

    "Xception":
        xception_preprocess,

    "InceptionV3":
        inception_v3_preprocess,

    "InceptionResNetV2":
        inception_resnet_v2_preprocess,

    "EfficientNetB0":
        None,

    "EfficientNetB7":
        None,

    "EfficientNetV2S":
        None,
     "ConvNeXtTiny": tf.keras.applications.convnext.preprocess_input,
    "MobileNetV3Large": tf.keras.applications.mobilenet_v3.preprocess_input
}

IMG_SIZE = (224, 224)

BATCH_SIZE = 16

AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomRotation(
        factor=0.08
    ),

    tf.keras.layers.RandomTranslation(
        height_factor=0.10,
        width_factor=0.10
    ),

    tf.keras.layers.RandomZoom(
        height_factor=(-0.15, 0.15),
        width_factor=(-0.15, 0.15)
    ),

    tf.keras.layers.RandomBrightness(
        factor=0.20
    )

], name="data_augmentation")

def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    )

    label = tf.cast(
        label,
        tf.float32
    )

    return image, label



In [ ]:
def get_preprocess_function(model_name):

    if model_name not in PREPROCESS_FUNCTIONS:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return PREPROCESS_FUNCTIONS[model_name]


def create_dataset(
    dataframe,
    model_name,
    training=False
):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            dataframe["path"].values,
            dataframe["gender"].values
        )
    )

    # Load images
    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    # Augmentation فقط برای train
    if training:

        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

        dataset = dataset.map(
            lambda image, label: (
                data_augmentation(
                    image,
                    training=True
                ),
                label
            ),
            num_parallel_calls=AUTOTUNE
        )

    preprocess_fn = get_preprocess_function(
        model_name
    )

    if preprocess_fn is not None:

        dataset = dataset.map(
            lambda image, label: (
                preprocess_fn(image),
                label
            ),
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset

In [11]:
#VGG16

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="VGG16",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="VGG16",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="VGG16",
    training=False
)

vgg16_preprocess

import tensorflow as tf

from tensorflow.keras.applications import VGG16

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)


MODEL_NAME = "VGG16"

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_VGG16_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_VGG16_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("classification")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)


cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)


plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME}'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# DenseNet121 model

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="DenseNet121",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="DenseNet121",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="DenseNet121",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import DenseNet121

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "DenseNet121"

base_model = DenseNet121(
        weights='C:/Users/ASUS/Desktop/j/ears/densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_DenseNet121_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_DenseNet121_model.keras"
)
y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)


cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EfficientNetB7 model

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="EfficientNetB7",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="EfficientNetB7",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="EfficientNetB7",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB7

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "EfficientNetB7"

base_model = EfficientNetB7(
        weights='C:/Users/ASUS/Desktop/j/ears/efficientnetb7_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_EfficientNetB7_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_EfficientNetB7_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [37]:
# EfficientNetB0 model

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="EfficientNetB0",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="EfficientNetB0",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="EfficientNetB0",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "EfficientNetB0"

base_model = EfficientNetB0(
        weights='C:/Users/ASUS/Desktop/j/ears/efficientnetb0_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_EfficientNetB0_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_EfficientNetB0_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [39]:
# Xception model

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="Xception",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="Xception",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="Xception",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import Xception

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "Xception"

base_model = Xception(
        weights='C:/Users/ASUS/Desktop/j/ears/xception_weights_tf_dim_ordering_tf_kernels_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_Xception_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_Xception_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [46]:
#InceptionResNetV2

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="InceptionResNetV2",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="InceptionResNetV2",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="InceptionResNetV2",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import InceptionResNetV2

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "InceptionResNetV2"


base_model = InceptionResNetV2(
        weights='C:/Users/ASUS/Desktop/j/ears/inception_resnet_v2_weights_tf_dim_ordering_tf_kernels_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_InceptionResNetV2_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_InceptionResNetV2_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()


import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [36]:
# ConvNeXt-Tiny

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="ConvNeXtTiny",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="ConvNeXtTiny",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="ConvNeXtTiny",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import ConvNeXtTiny

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "ConvNeXtTiny"

base_model = ConvNeXtTiny(
        weights='C:/Users/ASUS/Desktop/j/ears/convnext_tiny_notop.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_ConvNeXtTiny_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_ConvNeXtTiny_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [13]:
#MobileNetV3-Large

In [ ]:
train_dataset = create_dataset(
    train_df,
    model_name="MobileNetV3Large",
    training=True
)

validation_dataset = create_dataset(
    val_df,
    model_name="MobileNetV3Large",
    training=False
)

test_dataset = create_dataset(
    test_df,
    model_name="MobileNetV3Large",
    training=False
)

import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Large

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

MODEL_NAME = "MobileNetV3Large"

base_model = MobileNetV3Large(
        weights='C:/Users/ASUS/Desktop/j/ears/weights_mobilenet_v3_large_224_1.0_float_no_top_v2.h5',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        1,
        activation="sigmoid"
    )

], name=MODEL_NAME)

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

model.summary()

checkpoint = ModelCheckpoint(

    filepath=r"C:\Users\ASUS\Desktop\j\ears\best_MobileNetV3Large_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)


early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-7,

    verbose=1
)

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=50,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

model = tf.keras.models.load_model(

    r"C:\Users\ASUS\Desktop\j\ears\best_MobileNetV3Large_model.keras"
)

y_true = []
y_prob = []


for images, labels in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    ).flatten()

    y_true.extend(
        labels.numpy()
    )

    y_prob.extend(
        predictions
    )


y_true = np.array(y_true).astype(int)

y_prob = np.array(y_prob)

y_pred = (
    y_prob >= 0.5
).astype(int)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
    average='macro'
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0, average='macro'
)

auc_roc = roc_auc_score(
    y_true,
    y_prob
)

print("\n" + "=" * 60)

print(
    f"{MODEL_NAME} - TEST RESULTS"
)

print("=" * 60)

print(
    f"Accuracy   : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision  : {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall     : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"F1-score   : {f1:.4f} "
    f"({f1 * 100:.2f}%)"
)

print(
    f"AUC-ROC    : {auc_roc:.4f}"
)


print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Men",
            "Women"
        ],
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Men",
        "Women"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax
)

plt.title(
    f"Confusion Matrix - {MODEL_NAME}"
)

plt.tight_layout()

plt.show()

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)


plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"{MODEL_NAME} (AUC = {auc_roc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    f"ROC Curve - {MODEL_NAME}"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()

import matplotlib.pyplot as plt

train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_accuracy,
    label='Training Accuracy'
)

plt.plot(
    epochs_range,
    val_accuracy,
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.title(
    f'{MODEL_NAME} - Training and Validation Accuracy'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_range,
    train_loss,
    label='Training Loss'
)

plt.plot(
    epochs_range,
    val_loss,
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title(
    f'{MODEL_NAME} - Training and Validation Loss'
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()